# Module 3.1 — Univariate Time Series Analysis
## Listening to the Voice of a Single Instrument

---

### On the Nature of Financial Time

A price series is a biography written by the market. It records, tick by tick and bar by bar, the evolving consensus of millions of participants about what an asset is worth. But this biography is deceptive. It looks like a trajectory—something with direction, intention, momentum. Our pattern-recognizing brains see trends, cycles, support levels, channels. Some of these patterns are real. Most are not. The central project of time series analysis is to develop **rigorous tools for distinguishing signal from noise** in sequential data, and to understand the *statistical character* of the process that generates prices.

This distinction matters enormously. If a price series is a **random walk**, then past prices contain no information about future prices, and technical analysis is numerology. If the series has **mean-reverting** tendencies, then large deviations from the mean are opportunities, and you should fade extreme moves. If the series has **momentum** (autocorrelation in returns), then trends are real and you should follow them. If **volatility clusters** (periods of high volatility follow periods of high volatility), then you need models that capture this heteroscedasticity, or your risk estimates will be dangerously wrong half the time.

The tools in this module—stationarity tests, ACF/PACF analysis, ARIMA models, seasonal decomposition, and GARCH volatility models—are not merely statistical techniques. They are **lenses** through which you examine the character of a price process. Each lens reveals a different aspect: the ACF reveals memory, the ADF test reveals mean-reversion (or its absence), the GARCH model reveals the dynamics of fear and complacency. A quantitative trader who cannot perform these analyses is building strategies on assumptions they have never tested.

---

### Learning Objectives

By the end of this module, you will:

1. **Test** price and return series for stationarity using ADF and KPSS tests
2. **Interpret** autocorrelation (ACF) and partial autocorrelation (PACF) functions
3. **Fit and diagnose** ARMA and ARIMA models for financial returns
4. **Decompose** time series into trend, seasonal, and residual components
5. **Model volatility clustering** with GARCH and its variants
6. **Forecast** returns and volatility with proper confidence intervals
7. **Recognize** the limitations of each model when applied to financial data

---

### Module Contents

1. **Stationarity** — The assumption everything depends on, and why prices violate it
2. **Unit Root Tests** — ADF and KPSS: complementary diagnostics for a non-stationary world
3. **Autocorrelation & Partial Autocorrelation** — Reading the memory of a process
4. **ARMA and ARIMA Models** — Modeling the predictable structure in returns
5. **Seasonal Decomposition** — Extracting trend, seasonality, and residual
6. **GARCH Models** — When the variance itself has a story to tell
7. **Forecasting** — Predicting the future (and knowing how wrong you'll be)

---

*"The purpose of models is not to fit the data but to sharpen the questions." — Samuel Karlin*

*In finance, the sharpest question is always: what am I assuming?*

In [ ]:
# ========================================
# Initial Setup and Configuration
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime, timedelta
import warnings

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

PLOT_CONFIG = {
    'figure.figsize': (14, 7),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 100,
    'lines.linewidth': 2,
}
plt.rcParams.update(PLOT_CONFIG)

COLORS = {
    'price': '#1565C0',
    'returns': '#2E7D32',
    'volatility': '#C62828',
    'forecast': '#F57C00',
    'confidence': '#90CAF9',
    'trend': '#6A1B9A',
    'seasonal': '#00838F',
    'residual': '#455A64',
    'acf': '#1565C0',
}

print("=" * 80)
print(" " * 14 + "MODULE 3.1: UNIVARIATE TIME SERIES ANALYSIS")
print("=" * 80)
print(f"✓ Environment configured successfully")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n📚 Ready to listen to the voice of the market!\n")

---

## 1. Stationarity: The Foundation of Time Series Modeling

### 1.1 What Is Stationarity and Why Does It Matter?

A time series is **stationary** if its statistical properties—mean, variance, and autocovariance—do not change over time. Formally, a process $\{X_t\}$ is *weakly stationary* (or *covariance stationary*) if:

1. $\mathbb{E}[X_t] = \mu$ for all $t$ (constant mean)
2. $\text{Var}(X_t) = \sigma^2 < \infty$ for all $t$ (constant, finite variance)
3. $\text{Cov}(X_t, X_{t+h}) = \gamma(h)$ depends only on the lag $h$, not on $t$ (time-invariant autocovariance)

Why does stationarity matter? Because **virtually every classical time series model assumes it**. ARMA models, correlation analysis, spectral decomposition—all require stationarity to produce meaningful results. If you fit an ARMA model to a non-stationary series, the estimated coefficients are spurious: they describe a statistical relationship that does not exist and will not persist.

Here lies the fundamental tension in financial time series analysis: **prices are almost never stationary, but returns usually are** (or at least closer to it). A stock price that trends from $100 to $200 over a year has a time-varying mean and is unambiguously non-stationary. But the *daily returns* of that same stock—the percentage changes—tend to fluctuate around a near-zero mean with roughly constant (though time-varying in volatility) statistical properties. This is why quantitative finance overwhelmingly works with returns rather than prices.

### 1.2 Random Walks and Unit Roots

The simplest non-stationary model is the **random walk**:

$$P_t = P_{t-1} + \varepsilon_t, \quad \varepsilon_t \sim \text{WN}(0, \sigma^2)$$

Here $P_t$ is the price at time $t$ and $\varepsilon_t$ is white noise. The random walk has no mean-reversion: it drifts without bound, and its variance grows linearly with time ($\text{Var}(P_t) = t\sigma^2$). In the language of time series, the random walk has a **unit root**—its autoregressive coefficient is exactly 1. A series with a unit root is called **integrated of order 1** (denoted $I(1)$), and it becomes stationary after **differencing** once: $\Delta P_t = P_t - P_{t-1} = \varepsilon_t$, which is simply white noise.

The **random walk with drift** adds a constant:

$$P_t = \mu + P_{t-1} + \varepsilon_t$$

This produces a trending series. The drift $\mu$ represents the expected return per period. For stock prices, the *Efficient Market Hypothesis* in its weak form asserts that prices follow a random walk (possibly with drift), implying that past prices carry no information about future returns.

### 1.3 The ADF and KPSS Tests: Complementary Diagnostics

How do we formally test whether a series is stationary? Two tests dominate practice:

**Augmented Dickey-Fuller (ADF) Test**: Tests the null hypothesis that the series has a unit root (is non-stationary). The test regresses:

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t$$

and tests $H_0: \gamma = 0$ (unit root) against $H_1: \gamma < 0$ (stationary). If the test statistic is more negative than the critical value, we reject the null and conclude the series is stationary.

**KPSS Test**: Tests the *opposite* null—that the series **is** stationary. This makes ADF and KPSS natural complements:

| ADF Result | KPSS Result | Conclusion |
|-----------|-------------|------------|
| Reject $H_0$ | Fail to reject $H_0$ | **Stationary** |
| Fail to reject $H_0$ | Reject $H_0$ | **Non-stationary** |
| Reject $H_0$ | Reject $H_0$ | Trend-stationary (needs detrending) |
| Fail to reject $H_0$ | Fail to reject $H_0$ | Inconclusive |

Using both tests together provides much stronger evidence than either alone.

In [ ]:
# ========================================
# Stationarity Analysis
# ========================================

def generate_financial_series(n_days: int = 756, initial_price: float = 100.0,
                               annual_return: float = 0.08,
                               annual_vol: float = 0.20,
                               seed: int = 42) -> pd.DataFrame:
    """Generate a realistic stock price series using geometric Brownian motion,
    with volatility clustering to mimic real market dynamics."""
    rng = np.random.RandomState(seed)
    dt = 1 / 252
    n = n_days

    # Generate volatility clustering via a simple regime model
    vol_regime = np.ones(n)
    regime = 0  # 0=normal, 1=high vol
    for i in range(1, n):
        if regime == 0 and rng.random() < 0.02:
            regime = 1
        elif regime == 1 and rng.random() < 0.08:
            regime = 0
        vol_regime[i] = 1.0 if regime == 0 else 2.2

    daily_vol = annual_vol * np.sqrt(dt) * vol_regime
    daily_drift = (annual_return - 0.5 * (annual_vol * vol_regime)**2) * dt

    log_returns = daily_drift + daily_vol * rng.normal(0, 1, n)
    log_prices = np.log(initial_price) + np.cumsum(log_returns)
    prices = np.exp(log_prices)

    dates = pd.bdate_range(start='2022-01-03', periods=n)
    df = pd.DataFrame({
        'price': prices,
        'log_price': log_prices,
        'return': np.exp(log_returns) - 1,
        'log_return': log_returns,
        'vol_regime': vol_regime,
    }, index=dates)
    df.index.name = 'date'

    return df


@dataclass
class StationarityResult:
    """Container for stationarity test results."""
    series_name: str
    adf_statistic: float
    adf_pvalue: float
    adf_conclusion: str
    kpss_statistic: float
    kpss_pvalue: float
    kpss_conclusion: str
    joint_conclusion: str


def test_stationarity(series: pd.Series, name: str,
                       significance: float = 0.05) -> StationarityResult:
    """Run both ADF and KPSS tests and produce a joint conclusion."""
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit = kpss(series.dropna(), regression='c', nlags='auto')

    adf_reject = adf_p < significance
    kpss_reject = kpss_p < significance

    adf_conclusion = "Stationary (reject unit root)" if adf_reject else "Non-stationary (cannot reject unit root)"
    kpss_conclusion = "Non-stationary (reject stationarity)" if kpss_reject else "Stationary (cannot reject stationarity)"

    if adf_reject and not kpss_reject:
        joint = "STATIONARY"
    elif not adf_reject and kpss_reject:
        joint = "NON-STATIONARY"
    elif adf_reject and kpss_reject:
        joint = "TREND-STATIONARY"
    else:
        joint = "INCONCLUSIVE"

    return StationarityResult(
        series_name=name,
        adf_statistic=adf_stat, adf_pvalue=adf_p, adf_conclusion=adf_conclusion,
        kpss_statistic=kpss_stat, kpss_pvalue=kpss_p, kpss_conclusion=kpss_conclusion,
        joint_conclusion=joint,
    )


# --- Generate data and test stationarity ---

df = generate_financial_series(n_days=756)

series_to_test = {
    'Price Level': df['price'],
    'Log Price': df['log_price'],
    'Simple Returns': df['return'],
    'Log Returns': df['log_return'],
}

results = []
for name, series in series_to_test.items():
    result = test_stationarity(series, name)
    results.append(result)

print("=" * 80)
print("STATIONARITY ANALYSIS")
print("=" * 80)
print(f"\nData: {len(df)} trading days ({df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')})")
print(f"Initial price: ${df['price'].iloc[0]:.2f}, Final price: ${df['price'].iloc[-1]:.2f}")

print(f"\n{'Series':<18} {'ADF Stat':>10} {'ADF p':>8} {'KPSS Stat':>10} {'KPSS p':>8} {'Conclusion':>18}")
print("-" * 78)
for r in results:
    print(f"{r.series_name:<18} {r.adf_statistic:>10.4f} {r.adf_pvalue:>8.4f} "
          f"{r.kpss_statistic:>10.4f} {r.kpss_pvalue:>8.4f} {r.joint_conclusion:>18}")

print("\n💡 Prices are non-stationary (unit root); returns are stationary.")
print("   This is why we model returns, not prices.")

# --- Visualization ---

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Stationarity: Prices vs. Returns",
             fontsize=18, fontweight='bold', y=1.02)

# Price level
ax = axes[0, 0]
ax.plot(df.index, df['price'], color=COLORS['price'], linewidth=1.2)
ax.axhline(df['price'].mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: ${df["price"].mean():.1f}')
ax.set_title('Price Level (Non-Stationary)', fontsize=14)
ax.set_ylabel('Price ($)')
ax.legend()

# Log returns
ax = axes[0, 1]
ax.plot(df.index, df['log_return'], color=COLORS['returns'], linewidth=0.7, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(df['log_return'].mean(), color='red', linestyle='--', alpha=0.5,
           label=f'Mean: {df["log_return"].mean():.5f}')
ax.set_title('Log Returns (Stationary)', fontsize=14)
ax.set_ylabel('Log Return')
ax.legend()

# Rolling statistics for prices
ax = axes[1, 0]
window = 60
rolling_mean = df['price'].rolling(window).mean()
rolling_std = df['price'].rolling(window).std()
ax.plot(df.index, rolling_mean, color=COLORS['trend'], label=f'{window}-day Rolling Mean')
ax.fill_between(df.index, rolling_mean - 2*rolling_std, rolling_mean + 2*rolling_std,
                alpha=0.15, color=COLORS['trend'])
ax.set_title(f'Price: Rolling Mean ± 2σ (Drifting = Non-Stationary)', fontsize=14)
ax.set_ylabel('Price ($)')
ax.legend()

# Rolling statistics for returns
ax = axes[1, 1]
rolling_mean_r = df['log_return'].rolling(window).mean()
rolling_std_r = df['log_return'].rolling(window).std()
ax.plot(df.index, rolling_mean_r, color=COLORS['returns'], label=f'{window}-day Rolling Mean')
ax.fill_between(df.index, rolling_mean_r - 2*rolling_std_r, rolling_mean_r + 2*rolling_std_r,
                alpha=0.15, color=COLORS['returns'])
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'Returns: Rolling Mean ± 2σ (Stable = Stationary)', fontsize=14)
ax.set_ylabel('Log Return')
ax.legend()

plt.tight_layout()
plt.show()

---

## 2. Autocorrelation and the Memory of Markets

### 2.1 The Autocorrelation Function (ACF)

The **autocorrelation function** measures how a series correlates with lagged versions of itself. The ACF at lag $k$ is:

$$\rho(k) = \frac{\text{Cov}(X_t, X_{t+k})}{\text{Var}(X_t)} = \frac{\gamma(k)}{\gamma(0)}$$

For a purely random series (white noise), the ACF should be approximately zero at all non-zero lags—each observation is independent of its predecessors. Any significant deviation from zero indicates **serial dependence**, which is precisely the kind of structure a time series model can exploit.

In financial returns, the ACF typically shows:
- **Near-zero autocorrelation in returns** at most lags—consistent with market efficiency
- **Significant autocorrelation in squared returns** (or absolute returns)—evidence of volatility clustering
- **Small but significant negative autocorrelation at lag 1** in some markets—the bid-ask bounce effect, where alternating trades at the bid and ask create artificial negative correlation

### 2.2 The Partial Autocorrelation Function (PACF)

While the ACF measures the *total* correlation between $X_t$ and $X_{t-k}$, the **partial autocorrelation function** measures the correlation after removing the effects of intermediate lags. Formally, the PACF at lag $k$ is the coefficient $\phi_{kk}$ in the regression:

$$X_t = \phi_{k1} X_{t-1} + \phi_{k2} X_{t-2} + \cdots + \phi_{kk} X_{t-k} + \varepsilon_t$$

The PACF is the key tool for **identifying the order of AR models**:
- An **AR($p$)** process has a PACF that cuts off after lag $p$
- An **MA($q$)** process has an ACF that cuts off after lag $q$
- An **ARMA($p,q$)** process has both ACF and PACF that decay gradually

### 2.3 The Ljung-Box Test: Is There Any Structure Left?

The **Ljung-Box test** tests whether any of the first $m$ autocorrelations are significantly different from zero. Under the null hypothesis of no autocorrelation:

$$Q(m) = n(n+2) \sum_{k=1}^{m} \frac{\hat{\rho}(k)^2}{n-k} \sim \chi^2(m)$$

This test is essential for **model diagnostics**: after fitting an ARIMA model, the residuals should pass the Ljung-Box test (i.e., show no remaining autocorrelation). If they don't, the model has failed to capture all the serial structure in the data.

In [ ]:
# ========================================
# Autocorrelation Analysis
# ========================================

log_returns = df['log_return'].dropna()

# Compute ACF and PACF
n_lags = 40
acf_values = acf(log_returns, nlags=n_lags, fft=True)
pacf_values = pacf(log_returns, nlags=n_lags)
acf_sq = acf(log_returns**2, nlags=n_lags, fft=True)
acf_abs = acf(np.abs(log_returns), nlags=n_lags, fft=True)

# Ljung-Box test
lb_returns = acorr_ljungbox(log_returns, lags=[5, 10, 20, 40], return_df=True)
lb_squared = acorr_ljungbox(log_returns**2, lags=[5, 10, 20, 40], return_df=True)

print("=" * 80)
print("AUTOCORRELATION ANALYSIS")
print("=" * 80)

print("\nLjung-Box Test on Returns (H0: no autocorrelation):")
print(f"{'Lags':>6} {'Q-Stat':>10} {'p-value':>10} {'Significant?':>14}")
print("-" * 44)
for idx, row in lb_returns.iterrows():
    sig = "Yes ***" if row['lb_pvalue'] < 0.01 else ("Yes *" if row['lb_pvalue'] < 0.05 else "No")
    print(f"{idx:>6} {row['lb_stat']:>10.3f} {row['lb_pvalue']:>10.4f} {sig:>14}")

print("\nLjung-Box Test on Squared Returns (H0: no autocorrelation in variance):")
print(f"{'Lags':>6} {'Q-Stat':>10} {'p-value':>10} {'Significant?':>14}")
print("-" * 44)
for idx, row in lb_squared.iterrows():
    sig = "Yes ***" if row['lb_pvalue'] < 0.01 else ("Yes *" if row['lb_pvalue'] < 0.05 else "No")
    print(f"{idx:>6} {row['lb_stat']:>10.3f} {row['lb_pvalue']:>10.4f} {sig:>14}")

print("\n💡 Returns show little autocorrelation (markets are roughly efficient),")
print("   but squared returns are highly autocorrelated (volatility clusters).")

# --- Visualization: ACF/PACF plots ---

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Reading the Memory of the Market",
             fontsize=18, fontweight='bold', y=1.02)

conf_bound = 1.96 / np.sqrt(len(log_returns))

# ACF of returns
ax = axes[0, 0]
ax.bar(range(1, n_lags+1), acf_values[1:], color=COLORS['acf'], alpha=0.7, width=0.6)
ax.axhline(conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('ACF of Log Returns', fontsize=14)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')

# PACF of returns
ax = axes[0, 1]
ax.bar(range(1, n_lags+1), pacf_values[1:], color=COLORS['returns'], alpha=0.7, width=0.6)
ax.axhline(conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('PACF of Log Returns', fontsize=14)
ax.set_xlabel('Lag')
ax.set_ylabel('Partial Autocorrelation')

# ACF of squared returns (volatility clustering evidence)
ax = axes[1, 0]
ax.bar(range(1, n_lags+1), acf_sq[1:], color=COLORS['volatility'], alpha=0.7, width=0.6)
ax.axhline(conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('ACF of Squared Returns (Volatility Clustering)', fontsize=14)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')

# ACF of absolute returns
ax = axes[1, 1]
ax.bar(range(1, n_lags+1), acf_abs[1:], color=COLORS['forecast'], alpha=0.7, width=0.6)
ax.axhline(conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-conf_bound, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('ACF of Absolute Returns (Long Memory)', fontsize=14)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

---

## 3. ARMA and ARIMA Models

### 3.1 The ARMA Framework

The **Autoregressive Moving Average** model combines two intuitions about how a series might be generated:

- **AR($p$)**: The current value depends linearly on its $p$ most recent values. This captures **momentum** or **mean-reversion** depending on the sign and magnitude of the coefficients.

$$X_t = c + \phi_1 X_{t-1} + \phi_2 X_{t-2} + \cdots + \phi_p X_{t-p} + \varepsilon_t$$

- **MA($q$)**: The current value depends on the $q$ most recent **forecast errors**. This captures the market's reaction to news—how shocks propagate and dissipate.

$$X_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \cdots + \theta_q \varepsilon_{t-q}$$

The combined **ARMA($p,q$)** model is:

$$X_t = c + \sum_{i=1}^{p} \phi_i X_{t-i} + \varepsilon_t + \sum_{j=1}^{q} \theta_j \varepsilon_{t-j}$$

### 3.2 ARIMA: Handling Non-Stationarity

When the series is non-stationary but becomes stationary after differencing $d$ times, we use **ARIMA($p,d,q$)**, which applies the ARMA model to the $d$-th difference of the series. For stock prices, $d=1$ typically suffices (differencing prices gives returns). The "I" stands for "integrated"—the original series is the integral (cumulative sum) of the stationary differenced series.

### 3.3 Model Selection: AIC, BIC, and the Bias-Variance Tradeoff

How do you choose $p$, $d$, and $q$? The ACF/PACF plots provide initial guidance, but formal model selection uses information criteria:

$$\text{AIC} = -2\ln(\hat{L}) + 2k, \qquad \text{BIC} = -2\ln(\hat{L}) + k\ln(n)$$

where $\hat{L}$ is the maximized likelihood, $k$ is the number of parameters, and $n$ is the sample size. AIC tends to favor larger models (more parameters), while BIC penalizes complexity more heavily. In practice, we fit several candidate models and select the one with the lowest AIC or BIC.

A philosophical caution: the "best" model by AIC is not necessarily a *good* model. A model can be the least bad among a set of bad candidates. Always check the residuals—if they show remaining structure (autocorrelation, heteroscedasticity, non-normality), the model is incomplete, regardless of its information criterion score.

### 3.4 Seasonal Decomposition

Financial time series exhibit various periodic patterns: **intraday seasonality** (the U-shaped volume pattern, with high activity at the open and close), **day-of-week effects** (the historical Monday effect), and **calendar effects** (end-of-quarter window dressing, January effect). Seasonal decomposition separates a series into three components:

$$Y_t = T_t + S_t + R_t$$

where $T_t$ is the trend, $S_t$ is the seasonal component, and $R_t$ is the residual. Understanding these components helps avoid confusing calendar effects with genuine alpha signals.

In [ ]:
# ========================================
# ARIMA Modeling and Seasonal Decomposition
# ========================================

log_returns = df['log_return'].dropna()

# --- Grid search for best ARIMA model ---

print("=" * 80)
print("ARIMA MODEL SELECTION")
print("=" * 80)

candidate_orders = [
    (0,0,0), (1,0,0), (2,0,0), (3,0,0),
    (0,0,1), (0,0,2), (0,0,3),
    (1,0,1), (2,0,1), (1,0,2), (2,0,2),
]

model_results = []
for order in candidate_orders:
    try:
        model = ARIMA(log_returns, order=order)
        fitted = model.fit()
        lb_test = acorr_ljungbox(fitted.resid, lags=[10], return_df=True)
        model_results.append({
            'Order': f'ARIMA{order}',
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'Log-Lik': fitted.llf,
            'LB(10) p': lb_test['lb_pvalue'].values[0],
            'Params': len(fitted.params),
        })
    except Exception:
        pass

results_df = pd.DataFrame(model_results).sort_values('AIC')
print(f"\n{results_df.to_string(index=False)}")

best_order_str = results_df.iloc[0]['Order']
print(f"\n✓ Best model by AIC: {best_order_str}")

# --- Fit the best model and examine residuals ---

best_order = candidate_orders[results_df.index[0]] if results_df.index[0] < len(candidate_orders) else (1,0,1)
best_model = ARIMA(log_returns, order=best_order).fit()

print(f"\n{'=' * 80}")
print(f"BEST MODEL SUMMARY: ARIMA{best_order}")
print(f"{'=' * 80}")
print(best_model.summary().tables[1].as_text())

# --- Residual diagnostics ---

residuals = best_model.resid

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"ARIMA{best_order} Residual Diagnostics",
             fontsize=18, fontweight='bold', y=1.02)

# Residuals over time
ax = axes[0, 0]
ax.plot(residuals.index, residuals.values, color=COLORS['residual'], linewidth=0.7, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Residuals Over Time', fontsize=14)
ax.set_ylabel('Residual')

# Histogram of residuals
ax = axes[0, 1]
ax.hist(residuals, bins=50, density=True, color=COLORS['residual'], alpha=0.7, edgecolor='white')
x_range = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
        color='red', linewidth=2, label='Normal fit')
ax.set_title('Residual Distribution vs. Normal', fontsize=14)
ax.set_xlabel('Residual')
ax.legend()

# Q-Q plot
ax = axes[1, 0]
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('Q-Q Plot (Normal)', fontsize=14)
ax.get_lines()[0].set_color(COLORS['acf'])
ax.get_lines()[0].set_markersize(3)

# ACF of residuals
ax = axes[1, 1]
resid_acf = acf(residuals, nlags=30, fft=True)
ax.bar(range(1, 31), resid_acf[1:], color=COLORS['acf'], alpha=0.7, width=0.6)
conf = 1.96 / np.sqrt(len(residuals))
ax.axhline(conf, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(-conf, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('ACF of Residuals (Should Be White Noise)', fontsize=14)
ax.set_xlabel('Lag')

plt.tight_layout()
plt.show()

# --- Seasonal Decomposition ---

print("\n" + "=" * 80)
print("SEASONAL DECOMPOSITION")
print("=" * 80)

# Use a 5-day period to capture weekly seasonality in returns
abs_returns = np.abs(log_returns)
decomposition = seasonal_decompose(abs_returns, model='additive', period=5)

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle("Seasonal Decomposition of Absolute Returns\n(Weekly Periodicity)",
             fontsize=18, fontweight='bold', y=1.02)

components = [
    (decomposition.observed, 'Observed |r|', COLORS['price']),
    (decomposition.trend, 'Trend (Volatility Regime)', COLORS['trend']),
    (decomposition.seasonal, 'Weekly Seasonal Pattern', COLORS['seasonal']),
    (decomposition.resid, 'Residual', COLORS['residual']),
]

for ax, (data, title, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=0.9)
    ax.set_title(title, fontsize=13)
    ax.axhline(0, color='black', linewidth=0.3, alpha=0.5)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

print("\n💡 The trend component reveals volatility regimes.")
print("   The seasonal component captures any day-of-week effects.")

---

## 4. GARCH Models: When Volatility Has a Memory

### 4.1 The Phenomenon of Volatility Clustering

One of the most robust empirical facts in finance is **volatility clustering**: large returns (positive or negative) tend to be followed by large returns, and small returns tend to be followed by small returns. This was first documented by Mandelbrot (1963), who observed that "large changes tend to be followed by large changes—of either sign—and small changes tend to be followed by small changes." The ACF of squared returns we computed above is the formal evidence: it is significantly positive at many lags, meaning today's squared return is correlated with tomorrow's, next week's, and even next month's.

ARIMA models cannot capture this phenomenon because they model the **conditional mean** but assume **constant variance**. What we need is a model of the **conditional variance**—a model that says: given the recent history of large shocks, tomorrow's variance is expected to be high.

### 4.2 The GARCH(1,1) Model

The **Generalized Autoregressive Conditional Heteroscedasticity** model, introduced by Bollerslev (1986), is the workhorse volatility model. The GARCH(1,1) specification is:

$$r_t = \mu + \varepsilon_t, \quad \varepsilon_t = \sigma_t z_t, \quad z_t \sim N(0,1)$$

$$\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

where:
- $\sigma_t^2$ is the **conditional variance** at time $t$
- $\omega > 0$ is the long-run variance floor
- $\alpha \geq 0$ is the **ARCH coefficient**: how much yesterday's shock affects today's variance
- $\beta \geq 0$ is the **GARCH coefficient**: how much yesterday's variance persists into today
- $\alpha + \beta < 1$ ensures stationarity of the variance process

The **long-run (unconditional) variance** is:

$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}$$

The **persistence** $\alpha + \beta$ measures how slowly volatility reverts to its long-run level. In equity markets, this is typically 0.95–0.99, meaning volatility shocks are extremely persistent.

### 4.3 Asymmetric Volatility: The Leverage Effect

In equity markets, negative returns tend to increase volatility more than positive returns of equal magnitude. This asymmetry—called the **leverage effect** (because falling prices increase a firm's leverage ratio, making its equity riskier)—is not captured by the standard GARCH model, which treats positive and negative shocks symmetrically.

The **GJR-GARCH** (or Threshold GARCH) model adds an asymmetric term:

$$\sigma_t^2 = \omega + (\alpha + \gamma \mathbb{1}_{\{\varepsilon_{t-1}<0\}}) \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

where $\gamma > 0$ captures the additional impact of negative shocks. The **EGARCH** model of Nelson (1991) is another popular asymmetric specification.

### 4.4 Why GARCH Matters for Trading

Volatility is not just an academic curiosity—it is the **denominator of every risk-adjusted metric** you use:

- **Position sizing**: Kelly criterion and volatility targeting both require accurate volatility estimates
- **Options pricing**: Implied volatility is the market's GARCH forecast; comparing it to your own model is a trading strategy
- **Risk management**: VaR and Expected Shortfall depend critically on the volatility estimate
- **Signal normalization**: A mean-reversion z-score of 2 means something very different in a low-vol regime vs. a high-vol regime

Using historical (constant) volatility when the true volatility is time-varying is like using last year's weather to decide today's clothing. GARCH gives you a weather forecast.

In [ ]:
# ========================================
# GARCH Volatility Modeling
# ========================================

returns_pct = df['log_return'].dropna() * 100  # arch expects percentage returns

# --- Fit GARCH(1,1) ---

garch_model = arch_model(returns_pct, vol='Garch', p=1, q=1, mean='Constant', dist='normal')
garch_fit = garch_model.fit(disp='off')

print("=" * 80)
print("GARCH(1,1) MODEL")
print("=" * 80)
print(garch_fit.summary().tables[1].as_text())

omega = garch_fit.params['omega']
alpha = garch_fit.params['alpha[1]']
beta = garch_fit.params['beta[1]']
persistence = alpha + beta
long_run_var = omega / (1 - persistence) if persistence < 1 else float('inf')
long_run_vol_annual = np.sqrt(long_run_var * 252) if persistence < 1 else float('inf')
half_life = np.log(2) / np.log(1 / persistence) if persistence < 1 and persistence > 0 else float('inf')

print(f"\nKey Parameters:")
print(f"  ω (base variance):     {omega:.6f}")
print(f"  α (shock impact):      {alpha:.4f}")
print(f"  β (persistence):       {beta:.4f}")
print(f"  α + β (total persist): {persistence:.4f}")
print(f"  Long-run daily vol:    {np.sqrt(long_run_var):.4f}%")
print(f"  Long-run annual vol:   {long_run_vol_annual:.2f}%")
print(f"  Half-life of shocks:   {half_life:.1f} days")

# --- Fit GJR-GARCH for leverage effect ---

gjr_model = arch_model(returns_pct, vol='Garch', p=1, o=1, q=1, mean='Constant', dist='normal')
gjr_fit = gjr_model.fit(disp='off')

print(f"\n{'=' * 80}")
print("GJR-GARCH(1,1,1) MODEL — Asymmetric Volatility")
print(f"{'=' * 80}")
print(gjr_fit.summary().tables[1].as_text())

gamma = gjr_fit.params.get('gamma[1]', 0)
print(f"\n  γ (leverage effect): {gamma:.4f}")
if gamma > 0:
    print(f"  → Negative shocks increase variance by {gamma:.4f} more than positive shocks")
    print(f"  → This confirms the leverage effect in our simulated data")

# --- Extract conditional volatility ---

cond_vol = garch_fit.conditional_volatility
cond_vol_gjr = gjr_fit.conditional_volatility


def _get_regime_spans(mask):
    """Extract contiguous True spans from a boolean mask for axvspan."""
    spans = []
    in_span = False
    for idx, val in mask.items():
        if val and not in_span:
            start = idx
            in_span = True
        elif not val and in_span:
            spans.append((start, idx))
            in_span = False
    if in_span:
        spans.append((start, mask.index[-1]))
    return spans


# --- Visualization ---

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)
fig.suptitle("GARCH Volatility Analysis: When Variance Tells a Story",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Returns with conditional volatility bands
ax = axes[0]
ax.plot(returns_pct.index, returns_pct, color=COLORS['returns'], linewidth=0.5, alpha=0.6, label='Returns')
ax.plot(cond_vol.index, 2 * cond_vol, color=COLORS['volatility'], linewidth=1.2, label='±2σ (GARCH)')
ax.plot(cond_vol.index, -2 * cond_vol, color=COLORS['volatility'], linewidth=1.2)
ax.fill_between(cond_vol.index, -2 * cond_vol, 2 * cond_vol,
                alpha=0.08, color=COLORS['volatility'])
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Returns with GARCH(1,1) Conditional Volatility Bands', fontsize=14)
ax.set_ylabel('Return (%)')
ax.legend(loc='upper left')

# Panel 2: Conditional volatility comparison
ax = axes[1]
cond_vol_annual = cond_vol * np.sqrt(252)
cond_vol_gjr_annual = cond_vol_gjr * np.sqrt(252)
hist_vol = returns_pct.rolling(21).std() * np.sqrt(252)

ax.plot(hist_vol.index, hist_vol, color=COLORS['residual'], linewidth=1.0, alpha=0.5, label='21-day Historical Vol')
ax.plot(cond_vol_annual.index, cond_vol_annual, color=COLORS['volatility'], linewidth=1.5, label='GARCH(1,1)')
ax.plot(cond_vol_gjr_annual.index, cond_vol_gjr_annual, color=COLORS['forecast'],
        linewidth=1.5, linestyle='--', label='GJR-GARCH')
ax.set_title('Annualized Conditional Volatility: GARCH vs Historical', fontsize=14)
ax.set_ylabel('Annualized Volatility (%)')
ax.legend(loc='upper left')

# Highlight high-vol regimes from our simulation
high_vol_mask = df['vol_regime'].iloc[1:] > 1.5
for start, end in _get_regime_spans(high_vol_mask):
    ax.axvspan(start, end, alpha=0.08, color='red')

# Panel 3: News Impact Curve
ax = axes[2]
shock_range = np.linspace(-5, 5, 200)

# GARCH: symmetric
garch_impact = omega + alpha * shock_range**2 + beta * long_run_var

# GJR-GARCH: asymmetric
omega_gjr = gjr_fit.params['omega']
alpha_gjr = gjr_fit.params['alpha[1]']
beta_gjr = gjr_fit.params['beta[1]']
gamma_gjr = gjr_fit.params.get('gamma[1]', 0)
gjr_long_run = omega_gjr / (1 - alpha_gjr - beta_gjr - 0.5 * gamma_gjr) if (1 - alpha_gjr - beta_gjr - 0.5 * gamma_gjr) > 0 else long_run_var
gjr_impact = omega_gjr + (alpha_gjr + gamma_gjr * (shock_range < 0)) * shock_range**2 + beta_gjr * gjr_long_run

ax.plot(shock_range, garch_impact, color=COLORS['volatility'], linewidth=2, label='GARCH(1,1) — Symmetric')
ax.plot(shock_range, gjr_impact, color=COLORS['forecast'], linewidth=2, linestyle='--', label='GJR-GARCH — Asymmetric')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('News Impact Curve: How Shocks Affect Tomorrow\'s Variance', fontsize=14)
ax.set_xlabel('Shock (ε, in %)')
ax.set_ylabel('Next-Period Variance')
ax.legend()

plt.tight_layout()
plt.show()

---

## 5. Forecasting: Predicting the Future (and Quantifying Uncertainty)

### 5.1 The Limits of Prediction

Every forecasting model is a bet against the efficient market hypothesis. If markets are perfectly efficient, expected returns are unpredictable, and the best forecast of tomorrow's return is zero (plus a risk premium). The ARIMA model might find small, statistically significant autocorrelations—but whether these are economically significant after transaction costs is a separate question. Forecasting returns is hard. Forecasting volatility is easier, because volatility is persistent and clustered. This asymmetry explains why volatility trading (options, variance swaps) has a more robust theoretical and empirical foundation than directional trading.

### 5.2 The Forecasting Exercise

Below, we perform an out-of-sample forecasting exercise:

1. **Split** the data into training (first 80%) and test (last 20%) sets
2. **Fit** ARIMA and GARCH models on the training set
3. **Forecast** returns and volatility on the test set with confidence intervals
4. **Evaluate** forecast accuracy using RMSE, MAE, and hit rate

The critical discipline is that we never allow information from the test set to influence model fitting. This mirrors the real-world constraint: you can only trade on information available *now*, not information from the future.

In [ ]:
# ========================================
# Out-of-Sample Forecasting
# ========================================

# Train/test split
split_idx = int(len(returns_pct) * 0.8)
train = returns_pct.iloc[:split_idx]
test = returns_pct.iloc[split_idx:]

print("=" * 80)
print("OUT-OF-SAMPLE FORECASTING")
print("=" * 80)
print(f"Training set: {len(train)} observations ({train.index[0].strftime('%Y-%m-%d')} to {train.index[-1].strftime('%Y-%m-%d')})")
print(f"Test set:     {len(test)} observations ({test.index[0].strftime('%Y-%m-%d')} to {test.index[-1].strftime('%Y-%m-%d')})")

# --- ARIMA rolling 1-step forecasts ---

arima_forecasts = []
arima_conf_lower = []
arima_conf_upper = []

expanding_data = list(train.values)

for i in range(len(test)):
    try:
        model = ARIMA(expanding_data, order=best_order)
        fitted = model.fit()
        forecast = fitted.forecast(steps=1)
        conf = fitted.get_forecast(steps=1).conf_int(alpha=0.05)
        arima_forecasts.append(forecast[0])
        arima_conf_lower.append(conf[0, 0])
        arima_conf_upper.append(conf[0, 1])
    except Exception:
        arima_forecasts.append(0.0)
        arima_conf_lower.append(-2.0)
        arima_conf_upper.append(2.0)

    expanding_data.append(test.values[i])

arima_forecasts = np.array(arima_forecasts)

# --- GARCH volatility forecasts ---

garch_vol_forecasts = []

for i in range(len(test)):
    try:
        gm = arch_model(np.array(expanding_data[:split_idx + i]),
                        vol='Garch', p=1, q=1, mean='Constant', dist='normal')
        gf = gm.fit(disp='off', show_warning=False)
        vol_fc = gf.forecast(horizon=1)
        garch_vol_forecasts.append(np.sqrt(vol_fc.variance.values[-1, 0]))
    except Exception:
        garch_vol_forecasts.append(np.std(expanding_data[:split_idx + i]) if expanding_data else 1.0)

garch_vol_forecasts = np.array(garch_vol_forecasts)

# --- Evaluation ---

actual = test.values

# Return forecast metrics
rmse_arima = np.sqrt(np.mean((actual - arima_forecasts)**2))
mae_arima = np.mean(np.abs(actual - arima_forecasts))
rmse_naive = np.sqrt(np.mean(actual**2))  # naive forecast = 0
hit_rate = np.mean(np.sign(arima_forecasts) == np.sign(actual))

# Volatility forecast metrics
realized_vol = np.abs(actual)
rmse_garch_vol = np.sqrt(np.mean((realized_vol - garch_vol_forecasts)**2))
rmse_hist_vol = np.sqrt(np.mean((realized_vol - np.mean(np.abs(train.values)))**2))

print(f"\n{'Metric':<30} {'ARIMA':>12} {'Naive (zero)':>14}")
print("-" * 58)
print(f"{'RMSE (return):':<30} {rmse_arima:>12.4f} {rmse_naive:>14.4f}")
print(f"{'MAE (return):':<30} {mae_arima:>12.4f} {'':>14}")
print(f"{'Directional Hit Rate:':<30} {hit_rate:>11.1%} {'50.0%':>14}")

print(f"\n{'Metric':<30} {'GARCH':>12} {'Historical':>14}")
print("-" * 58)
print(f"{'RMSE (volatility):':<30} {rmse_garch_vol:>12.4f} {rmse_hist_vol:>14.4f}")

print("\n💡 Return forecasts barely beat naive zero-forecast (markets are efficient).")
print("   Volatility forecasts substantially beat historical average (vol is predictable).")

# --- Visualization ---

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)
fig.suptitle("Out-of-Sample Forecast Evaluation",
             fontsize=18, fontweight='bold', y=1.01)

# Panel 1: Return forecasts vs actual
ax = axes[0]
ax.plot(test.index, actual, color=COLORS['returns'], linewidth=0.7, alpha=0.7, label='Actual Returns')
ax.plot(test.index, arima_forecasts, color=COLORS['forecast'], linewidth=1.2, label=f'ARIMA{best_order} Forecast')
ax.fill_between(test.index, arima_conf_lower, arima_conf_upper,
                alpha=0.12, color=COLORS['confidence'], label='95% CI')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Return Forecasts vs. Actual', fontsize=14)
ax.set_ylabel('Return (%)')
ax.legend(loc='upper left')

# Panel 2: Volatility forecasts vs realized
ax = axes[1]
ax.plot(test.index, realized_vol, color=COLORS['residual'], linewidth=0.7, alpha=0.5, label='|Actual Return|')
ax.plot(test.index, garch_vol_forecasts, color=COLORS['volatility'], linewidth=1.5, label='GARCH Forecast σ')
ax.axhline(np.mean(np.abs(train.values)), color=COLORS['forecast'], linestyle='--',
           linewidth=1, alpha=0.6, label='Historical Avg |r|')
ax.set_title('Volatility Forecast: GARCH vs. Realized', fontsize=14)
ax.set_ylabel('Volatility (%)')
ax.legend(loc='upper left')

# Panel 3: Cumulative forecast error
ax = axes[2]
arima_errors = np.cumsum((actual - arima_forecasts)**2)
naive_errors = np.cumsum(actual**2)
ax.plot(test.index, arima_errors, color=COLORS['forecast'], linewidth=1.5, label=f'ARIMA{best_order} Cum. Squared Error')
ax.plot(test.index, naive_errors, color=COLORS['residual'], linewidth=1.5, linestyle='--', label='Naive Cum. Squared Error')
ax.set_title('Cumulative Squared Forecast Error (Lower = Better)', fontsize=14)
ax.set_ylabel('Cum. Squared Error')
ax.set_xlabel('Date')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


# ========================================
# Module Summary
# ========================================

print("\n" + "=" * 80)
print(" " * 18 + "MODULE 3.1 — SUMMARY")
print("=" * 80)
print("""
Key Concepts Covered:

  1. STATIONARITY
     • Prices are non-stationary; returns are (approximately) stationary
     • ADF tests for unit roots; KPSS tests for stationarity — use both
     • Differencing transforms I(d) processes to I(0)

  2. AUTOCORRELATION
     • ACF reveals serial dependence; PACF isolates direct effects
     • Returns show weak autocorrelation (efficiency)
     • Squared returns show strong autocorrelation (volatility clustering)
     • Ljung-Box test formally tests for remaining structure

  3. ARIMA MODELS
     • AR captures persistence; MA captures shock propagation
     • Model selection via AIC/BIC, validated by residual diagnostics
     • Seasonal decomposition separates trend, seasonal, and residual

  4. GARCH MODELS
     • Volatility clusters: large shocks beget large shocks
     • GARCH(1,1) captures conditional heteroscedasticity
     • GJR-GARCH captures the leverage effect (asymmetric response)
     • Volatility is far more predictable than returns

  5. FORECASTING
     • Return forecasts barely beat zero (markets are efficient)
     • Volatility forecasts substantially beat historical averages
     • Out-of-sample testing is the only honest evaluation

  The deepest lesson: the market's first moment (returns) is nearly
  unpredictable, but its second moment (volatility) is highly forecastable.
  This asymmetry is the foundation of volatility trading, risk management,
  and every strategy that uses position sizing.
""")
print("=" * 80)
print("\n📚 Next: Module 3.2 — Multivariate Time Series\n")